# 04 Data quality checks

Each check counts the rows that **violate** a rule. Zero violations means PASS.

- `critical` checks stop the pipeline (the job fails) when they find violations.
- `warning` checks are recorded but do not stop the pipeline (for example genuine missing months).

Results are appended to `gold_dq_results`, so you keep a history of every run.

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA air_quality")

DataFrame[]

In [0]:
# (name, severity, SQL that returns ONE number = count of violations, description)
CHECKS = [
    ("bronze_silver_rowcount_match", "critical",
     "SELECT ABS((SELECT COUNT(*) FROM bronze_air_pollution) - (SELECT COUNT(*) FROM silver_air_pollution))",
     "No rows lost or duplicated between bronze and silver"),

    ("no_duplicate_month_pollutant", "critical",
     "SELECT COUNT(*) FROM (SELECT reading_month, pollutant FROM silver_air_pollution GROUP BY 1, 2 HAVING COUNT(*) > 1)",
     "One row per month and pollutant"),

    ("dates_parsed", "critical",
     "SELECT COUNT(*) FROM silver_air_pollution WHERE reading_month IS NULL",
     "Every date string was parsed"),

    ("dates_are_first_of_month", "critical",
     "SELECT COUNT(*) FROM silver_air_pollution WHERE DAY(reading_month) <> 1",
     "Monthly data should be dated on the 1st"),

    ("no_negative_concentration", "critical",
     "SELECT COUNT(*) FROM silver_air_pollution WHERE concentration < 0",
     "A concentration can not be negative"),

    ("pollutants_exist_in_dimension", "critical",
     "SELECT COUNT(*) FROM silver_air_pollution s LEFT JOIN gold_dim_pollutant p ON s.pollutant = p.pollutant_code WHERE p.pollutant_code IS NULL",
     "Every pollutant in the data is described in gold_dim_pollutant"),

    ("fact_matches_silver", "critical",
     "SELECT ABS((SELECT COUNT(*) FROM gold_fact_air_quality) - (SELECT COUNT(*) FROM silver_air_pollution WHERE is_missing = FALSE))",
     "Fact table holds exactly the non-missing silver rows"),

    ("missing_values", "warning",
     "SELECT COUNT(*) FROM silver_air_pollution WHERE is_missing",
     "Months without a measurement (see gold_dq_completeness)"),

    ("extreme_values_z_gt_3", "warning",
     """SELECT COUNT(*) FROM (
          SELECT concentration,
                 AVG(concentration)    OVER (PARTITION BY pollutant) AS mean_c,
                 STDDEV(concentration) OVER (PARTITION BY pollutant) AS sd_c
          FROM silver_air_pollution
          WHERE is_missing = FALSE
        ) x
        WHERE sd_c > 0 AND ABS(concentration - mean_c) / sd_c > 3""",
     "Values more than 3 standard deviations from the pollutant mean (could be real events, review them)"),
]

In [0]:
%sql
-- How complete is each pollutant?
CREATE OR REPLACE TABLE gold_dq_completeness AS
SELECT
  pollutant,
  COUNT(*)                                          AS expected_months,
  SUM(CASE WHEN is_missing THEN 0 ELSE 1 END)       AS months_with_data,
  ROUND(100.0 * SUM(CASE WHEN is_missing THEN 0 ELSE 1 END) / COUNT(*), 1) AS pct_complete
FROM silver_air_pollution
GROUP BY pollutant

num_affected_rows,num_inserted_rows


In [0]:
from datetime import datetime

run_ts = datetime.now()
rows = []
for name, severity, sql, description in CHECKS:
    violations = int(spark.sql(sql).first()[0])
    if violations == 0:
        status = "PASS"
    else:
        status = "FAIL" if severity == "critical" else "WARN"
    rows.append((run_ts, name, severity, status, violations, description))

results = spark.createDataFrame(
    rows,
    "run_ts timestamp, check_name string, severity string, status string, violations long, description string",
)
results.write.format("delta").mode("append").saveAsTable("gold_dq_results")

display(results)

run_ts,check_name,severity,status,violations,description
2026-10-22T06:07:02.874Z,bronze_silver_rowcount_match,critical,PASS,0,No rows lost or duplicated between bronze and silver
2026-10-22T06:07:02.874Z,no_duplicate_month_pollutant,critical,PASS,0,One row per month and pollutant
2026-10-22T06:07:02.874Z,dates_parsed,critical,PASS,0,Every date string was parsed
2026-10-22T06:07:02.874Z,dates_are_first_of_month,critical,PASS,0,Monthly data should be dated on the 1st
2026-10-22T06:07:02.874Z,no_negative_concentration,critical,PASS,0,A concentration can not be negative
2026-10-22T06:07:02.874Z,pollutants_exist_in_dimension,critical,PASS,0,Every pollutant in the data is described in gold_dim_pollutant
2026-10-22T06:07:02.874Z,fact_matches_silver,critical,PASS,0,Fact table holds exactly the non-missing silver rows
2026-10-22T06:07:02.874Z,missing_values,warning,WARN,34,Months without a measurement (see gold_dq_completeness)
2026-10-22T06:07:02.874Z,extreme_values_z_gt_3,warning,WARN,11,"Values more than 3 standard deviations from the pollutant mean (could be real events, review them)"


In [0]:
failed = [r for r in rows if r[3] == "FAIL"]
if failed:
    names = ", ".join(r[1] for r in failed)
    raise Exception(f"Critical data quality checks failed: {names}")
print("All critical checks passed.")

All critical checks passed.
